In [0]:
MERGE INTO analisis_tmdb.gold.fact_movies t

USING (
    SELECT
        dm.movie_key,
        dg.genre_key,
        dd.date_key,
        m.popularity,
        m.vote_average,
        m.vote_count,
        (m.vote_average * m.vote_count) AS rating_weighted,
        m.ingestion_timestamp

    FROM analisis_tmdb.silver.movies m
    JOIN analisis_tmdb.silver.movie_genres mg
        ON m.movie_id = mg.movie_id

    JOIN analisis_tmdb.gold.dim_movies dm
        ON m.movie_id = dm.movie_id

    JOIN analisis_tmdb.gold.dim_genres dg
        ON mg.genre_id = dg.genre_id

    JOIN analisis_tmdb.gold.dim_date dd
        ON CAST(DATE_FORMAT(m.release_date, 'yyyyMMdd') AS INT) = dd.date_key
) s

ON t.movie_key = s.movie_key
AND t.genre_key = s.genre_key
AND t.date_key = s.date_key

WHEN MATCHED 
AND t.ingestion_timestamp < s.ingestion_timestamp THEN UPDATE SET
    popularity = s.popularity,
    vote_average = s.vote_average,
    vote_count = s.vote_count,
    rating_weighted = s.rating_weighted,
    ingestion_timestamp = s.ingestion_timestamp

WHEN NOT MATCHED THEN INSERT *
;